# Jour 1 · Les fondations des agents vocaux

**Formation Agents vocaux de 0 à Z · AI&ML Factory**

Ce notebook accompagne la journée 1. Chaque module a sa section, qui peut s'exécuter seule : si vous prenez du retard ou si une cellule plante, relancez simplement la **cellule de reprise** en tête de la section suivante.

| Section | Module | Statut |
|---|---|---|
| 0 | Configuration | disponible |
| 1 | Panorama et architecture d'un agent vocal | disponible |
| 2 | Le son numérique | disponible |
| 3 | Le transport temps réel | disponible |

**Fil rouge** : Awa, l'assistante commerciale vocale de Baobab Software (entreprise fictive). Elle qualifie un prospect, vérifie l'agenda et réserve un rendez-vous.

**Niveaux d'exercices**
- **[Guidé]** : code à compléter aux endroits marqués `# TODO`, avec un indice.
- **[Avancé]** : consigne ouverte, pour aller plus loin.

## 0. Configuration

Exécutez cette cellule une fois au début de la journée. Les Modules 1 à 3 fonctionnent **hors ligne**, sans clé API, sauf la section 1.5 (optionnelle).

In [ ]:
# Installation (décommentez si nécessaire, par exemple sur Colab)
# !pip install -q numpy scipy matplotlib websockets openai

import asyncio, time, json, re, inspect, os, random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# Palette de la formation
NAVY, TEAL, AMBER, GRAY, LIGHT = "#19233E", "#1E8C95", "#F2A93B", "#C9CFDB", "#9CCFD3"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

CLE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print("Configuration prête.", "Clé OpenAI détectée." if CLE_OPENAI else "Mode hors ligne (aucune clé API détectée).")

In [ ]:
# Outils communs à toute la journée

def chronogramme(evenements, titre="", fin_phrase=None, xmax=None, ordre=None):
    """Trace un chronogramme. evenements = liste de (couloir, debut_s, fin_s, etiquette, couleur)."""
    couloirs = list(ordre) if ordre else []
    for c, *_ in evenements:
        if c not in couloirs:
            couloirs.append(c)
    fig, ax = plt.subplots(figsize=(10, 0.55 * len(couloirs) + 1.2))
    for c, d, f, lab, col in evenements:
        y = len(couloirs) - 1 - couloirs.index(c)
        ax.barh(y, max(f - d, 0.01), left=d, height=0.6, color=col)
        if lab:
            ax.text(d + 0.02, y, lab, va="center", fontsize=8.5,
                    color="white" if col in (NAVY, TEAL) else "black")
    ax.set_yticks(range(len(couloirs)))
    ax.set_yticklabels(couloirs[::-1])
    ax.set_xlabel("temps (s)")
    if fin_phrase is not None:
        ax.axvline(fin_phrase, color="#C0392B", ls="--", lw=1)
    if xmax:
        ax.set_xlim(0, xmax)
    ax.set_title(titre, loc="left", fontweight="bold")
    plt.tight_layout()
    plt.show()

print("Outils chargés : chronogramme()")

---
## 1. Panorama et architecture d'un agent vocal

**Objectifs**
1. Décrire les briques d'un agent vocal et leur rôle.
2. Comprendre pourquoi le **streaming** change tout pour la latence perçue.
3. Manipuler un **budget de latence** et lire une médiane et un p95.
4. Comprendre le **cycle d'un appel d'outil** : le LLM demande, notre code exécute.

**Durée** : 30 minutes, puis le quiz (10 minutes).

In [ ]:
# Cellule de reprise du Module 1 : relancez-la si vous arrivez directement ici
try:
    chronogramme
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
print("Module 1 prêt.")

### 1.0 Rappel : l'anatomie d'un agent vocal

Un agent vocal en **cascade** enchaîne trois spécialistes :

| Brique | Rôle | Entrée → sortie | Exemples |
|---|---|---|---|
| **STT** (Speech-to-Text) | écouter | audio → texte | Deepgram, Whisper, AssemblyAI |
| **LLM** + outils | comprendre et agir | texte → texte ou appel d'outil | OpenAI, Anthropic, Gemini |
| **TTS** (Text-to-Speech) | répondre | texte → audio | ElevenLabs, Cartesia |

Autour d'eux : le **transport** (WebRTC, SIP) et la **détection de fin de tour** (VAD + modèle sémantique), qui décide quand l'agent a le droit de parler.

Un modèle **speech-to-speech** (OpenAI Realtime, Gemini Live) remplace les trois briques par un seul modèle : audio → audio.

La cellule suivante dessine le schéma.

In [ ]:
def schema_anatomie():
    etapes = [("Utilisateur", GRAY), ("Transport", LIGHT), ("Fin de tour", AMBER), ("STT", LIGHT),
              ("LLM + outils", NAVY), ("TTS", LIGHT), ("Réponse", GRAY)]
    fig, ax = plt.subplots(figsize=(11, 2.2))
    for i, (nom, col) in enumerate(etapes):
        ax.add_patch(FancyBboxPatch((i * 1.5, 0), 1.2, 0.8, boxstyle="round,pad=0.03", color=col))
        ax.text(i * 1.5 + 0.6, 0.4, nom, ha="center", va="center", fontsize=9.5, fontweight="bold",
                color="white" if col == NAVY else "black")
        if i < len(etapes) - 1:
            ax.annotate("", xy=(i * 1.5 + 1.48, 0.4), xytext=(i * 1.5 + 1.22, 0.4),
                        arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.5))
    ax.annotate("", xy=(0.6, -0.05), xytext=(9.6, -0.05),
                arrowprops=dict(arrowstyle="->", color=TEAL, ls="--", connectionstyle="arc3,rad=-0.12"))
    ax.text(5.1, -1.05, "en boucle, à chaque tour de parole", ha="center", color=TEAL, fontsize=9, style="italic")
    ax.set_xlim(-0.2, 10.4); ax.set_ylim(-1.3, 1); ax.axis("off")
    plt.show()

schema_anatomie()

### 1.1 Un agent simulé

On remplace les vraies briques par des **fonctions factices** qui attendent un certain temps (`asyncio.sleep`). Les durées sont **illustratives** mais réalistes. Cela permet de comprendre l'architecture sans clé API.

Le prospect vient de dire : *« Je voudrais un rendez-vous mardi. »* On chronomètre à partir de la **fin de sa phrase**.

In [ ]:
# Durées simulées (en secondes) : vous pourrez les modifier
LAT = {
    "fin_de_tour": 0.35,   # attente avant de déclarer que le prospect a fini
    "stt_final": 0.20,     # transcription finale après la fin de la parole
    "llm_ttft": 0.40,      # temps jusqu'au premier token du LLM
    "llm_token": 0.03,     # temps par token suivant
    "tts_ttfb": 0.20,      # temps jusqu'au premier son du TTS
    "tts_par_mot": 0.04,   # temps de synthèse par mot (mode non streamé)
}
REPONSE = "Bien sûr. Je regarde les disponibilités de mardi pour vous. J'ai 10 h ou 15 h, lequel vous convient ?"

async def stt(audio):
    await asyncio.sleep(LAT["stt_final"])
    return "Je voudrais un rendez-vous mardi."

async def llm_complet(texte):
    tokens = REPONSE.split()
    await asyncio.sleep(LAT["llm_ttft"] + LAT["llm_token"] * (len(tokens) - 1))
    return REPONSE

async def tts_complet(texte):
    await asyncio.sleep(LAT["tts_ttfb"] + LAT["tts_par_mot"] * len(texte.split()))
    return b"audio"

async def agent_sequentiel():
    """Chaque brique attend que la précédente ait totalement fini."""
    t0 = time.perf_counter(); ev = []
    def t(): return time.perf_counter() - t0
    await asyncio.sleep(LAT["fin_de_tour"]); ev.append(("Fin de tour", 0, t(), "attente", AMBER))
    a = t(); texte = await stt(b"..."); ev.append(("STT", a, t(), "transcription", LIGHT))
    a = t(); rep = await llm_complet(texte); ev.append(("LLM", a, t(), "réponse complète", NAVY))
    a = t(); await tts_complet(rep); ev.append(("TTS", a, t(), "synthèse complète", TEAL))
    premier_son = t(); ev.append(("Haut-parleur", premier_son, premier_son + 2.5, "Awa parle", TEAL))
    return premier_son, ev

premier_son_seq, ev_seq = await agent_sequentiel()
print(f"Mode séquentiel : premier son {premier_son_seq:.2f} s après la fin de la phrase.")
chronogramme(ev_seq, "Agent séquentiel", fin_phrase=0, xmax=5)

**[Guidé] Exercice 1.1** : le prospect raccroche si le silence dépasse 1,5 s. Calculez le pourcentage de dépassement du mode séquentiel par rapport à ce seuil.

*Indice* : `(valeur - seuil) / seuil * 100`.

In [ ]:
SEUIL = 1.5
depassement = None  # TODO : calculez le dépassement en pourcentage à partir de premier_son_seq et SEUIL

if depassement is None:
    print("Complétez la cellule ci-dessus.")
else:
    print(f"Le mode séquentiel dépasse le seuil de {depassement:.0f} %.")

### 1.2 Séquentiel contre streaming

En **streaming**, chaque brique travaille au fil de l'eau :
- le STT a déjà transcrit pendant que le prospect parlait ;
- le LLM envoie ses tokens un par un ;
- dès qu'une **phrase** est complète, le TTS commence à la dire, pendant que le LLM continue.

C'est ce que font LiveKit Agents ou Pipecat. Mesurons le gain sur le **premier son**.

In [ ]:
async def llm_stream(texte):
    """Génère la réponse token par token."""
    await asyncio.sleep(LAT["llm_ttft"])
    for i, tok in enumerate(REPONSE.split()):
        if i:
            await asyncio.sleep(LAT["llm_token"])
        yield tok

async def tts_phrase(phrase):
    await asyncio.sleep(LAT["tts_ttfb"])   # en streaming, seul le premier son compte

def phrase_complete(buffer):
    """Renvoie True quand le buffer se termine par une ponctuation de fin de phrase."""
    return buffer.rstrip().endswith((".", "?", "!"))

async def agent_streaming():
    t0 = time.perf_counter(); ev = []
    def t(): return time.perf_counter() - t0
    await asyncio.sleep(LAT["fin_de_tour"]); ev.append(("Fin de tour", 0, t(), "attente", AMBER))
    a = t(); await asyncio.sleep(0.05); ev.append(("STT", a, t(), "", LIGHT))  # déjà transcrit en grande partie
    premier = {}

    async def dire(phrase, debut):
        """Le TTS tourne en parallèle : le LLM continue de générer pendant ce temps."""
        await tts_phrase(phrase)
        if not premier:
            premier["t"] = t(); ev.append(("TTS", debut, premier["t"], "", TEAL))

    a = t(); buffer, taches = "", []
    async for tok in llm_stream("..."):
        buffer += tok + " "
        if phrase_complete(buffer):
            phrase, buffer = buffer.strip(), ""
            taches.append(asyncio.create_task(dire(phrase, t())))
    ev.append(("LLM", a, t(), "tokens en streaming", NAVY))
    await asyncio.gather(*taches)
    premier_son = premier["t"]
    ev.append(("Haut-parleur", premier_son, premier_son + 2.5, "Awa parle", TEAL))
    return premier_son, ev

premier_son_str, ev_str = await agent_streaming()
print(f"Séquentiel : {premier_son_seq:.2f} s   |   Streaming : {premier_son_str:.2f} s   |   "
      f"gain : {premier_son_seq - premier_son_str:.2f} s")
chronogramme(ev_str, "Agent en streaming", fin_phrase=0, xmax=5,
             ordre=["Fin de tour", "STT", "LLM", "TTS", "Haut-parleur"])

**À observer** : en streaming, le LLM continue de générer pendant qu'Awa parle déjà. Le **premier son** arrive bien plus tôt, alors que le temps total de génération est le même.

**[Guidé] Exercice 1.2** : la première phrase de la réponse, « Bien sûr. », est très courte, ce qui aide beaucoup. Remplacez-la par une longue phrase sans ponctuation au début et relancez l'agent en streaming. Que devient le gain ?

**[Avancé] Exercice 1.2 bis** : améliorez `phrase_complete` pour envoyer au TTS un morceau dès qu'il atteint **8 mots** ou une virgule, même sans point. Mesurez le gain et discutez du risque (intonation coupée au milieu d'une phrase).

In [ ]:
# [Guidé] Exercice 1.2 : modifiez la réponse et relancez
REPONSE_ORIGINALE = REPONSE
REPONSE = REPONSE_ORIGINALE  # TODO : remplacez par une réponse dont la première phrase est longue

premier_son_long, _ = await agent_streaming()
print(f"Premier son avec cette réponse : {premier_son_long:.2f} s")
REPONSE = REPONSE_ORIGINALE  # on restaure la réponse pour la suite

### 1.3 Le budget de latence

Le **délai perçu** va de la fin de la phrase du prospect au premier son d'Awa. On l'additionne brique par brique. Voici des options **illustratives** pour chaque brique.

In [ ]:
OPTIONS = {
    "reseau":      {"serveur lointain": 0.25, "serveur proche": 0.08},
    "fin_de_tour": {"prudente": 0.60, "équilibrée": 0.30, "agressive": 0.15},
    "stt":         {"batch": 0.50, "streaming": 0.15},
    "llm":         {"grand modèle": 0.70, "modèle rapide": 0.30},
    "tts":         {"haute qualité": 0.35, "faible latence": 0.12},
}

def budget(config):
    """config = {brique: option}. Renvoie le délai perçu total en secondes."""
    return sum(OPTIONS[b][o] for b, o in config.items())

NOMS = {"reseau": "Réseau", "fin_de_tour": "Fin de tour", "stt": "STT", "llm": "LLM", "tts": "TTS"}

def tracer_budget(config, titre):
    fig, ax = plt.subplots(figsize=(10, 1.6)); gauche = 0
    couleurs = {"reseau": GRAY, "fin_de_tour": AMBER, "stt": LIGHT, "llm": NAVY, "tts": TEAL}
    for b, o in config.items():
        v = OPTIONS[b][o]
        ax.barh(0, v, left=gauche, color=couleurs[b])
        ax.text(gauche + v / 2, 0, f"{NOMS[b]}\n{int(v*1000)} ms", ha="center", va="center", fontsize=8,
                color="white" if b in ("llm",) else "black")
        gauche += v
    ax.axvline(1.0, color="#C0392B", ls="--"); ax.text(1.01, 0.42, "cible 1 s", color="#C0392B", fontsize=8)
    ax.set_xlim(0, max(gauche, 1.0) * 1.1); ax.set_yticks([]); ax.set_xlabel("secondes")
    ax.set_title(f"{titre} : {gauche:.2f} s", loc="left", fontweight="bold"); plt.tight_layout(); plt.show()

config_depart = {"reseau": "serveur lointain", "fin_de_tour": "prudente", "stt": "batch",
                 "llm": "grand modèle", "tts": "haute qualité"}
tracer_budget(config_depart, "Configuration de départ")

**[Guidé] Exercice 1.3** : trouvez une configuration **sous 1 seconde**, avec une contrainte : la fin de tour ne doit **pas** être « agressive », sinon Awa coupe la parole aux prospects qui hésitent.

*Indice* : quel poste pèse le plus dans la configuration de départ ?

In [ ]:
ma_config = {
    "reseau": "serveur lointain",   # TODO
    "fin_de_tour": "prudente",      # TODO (pas "agressive")
    "stt": "batch",                 # TODO
    "llm": "grand modèle",          # TODO
    "tts": "haute qualité",         # TODO
}
total = budget(ma_config)
ok = total < 1.0 and ma_config["fin_de_tour"] != "agressive"
print(f"Total : {total:.2f} s  ->  {'objectif atteint' if ok else 'pas encore'}")
tracer_budget(ma_config, "Ma configuration")

**[Avancé] Exercice 1.3 bis : médiane et p95.** Dans la réalité, chaque brique **varie** d'un appel à l'autre. Une moyenne sous 1 s ne suffit pas : ce sont les appels lents que les prospects retiennent. Le **p95** est la valeur sous laquelle tombent 95 % des appels.

La cellule simule 1 000 appels avec une variation aléatoire. Lisez la médiane et le p95, puis cherchez une configuration dont le **p95** passe sous 1,2 s.

In [ ]:
rng = np.random.default_rng(42)

def simuler_appels(config, n=1000, dispersion=0.35):
    """Chaque brique suit une loi log-normale centrée sur sa valeur nominale."""
    totaux = np.zeros(n)
    for b, o in config.items():
        totaux += OPTIONS[b][o] * rng.lognormal(mean=0, sigma=dispersion, size=n)
    return totaux

totaux = simuler_appels(ma_config)
med, p95 = np.median(totaux), np.percentile(totaux, 95)
plt.figure(figsize=(9, 2.8))
plt.hist(totaux, bins=50, color=LIGHT)
plt.axvline(med, color=NAVY, label=f"médiane {med:.2f} s")
plt.axvline(p95, color="#C0392B", ls="--", label=f"p95 {p95:.2f} s")
plt.xlabel("délai perçu (s)"); plt.legend(); plt.title("1 000 appels simulés", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Médiane : {med:.2f} s   |   p95 : {p95:.2f} s   |   appels au-delà de 1,5 s : {np.mean(totaux > 1.5):.1%}")

### 1.4 Simuler un appel d'outil

Le cycle du **function calling** :
1. notre code envoie au LLM le texte, le contexte et la **liste des outils** (sous forme de schémas JSON) ;
2. le LLM répond soit par du texte, soit par une **demande d'appel d'outil** ;
3. **notre code** exécute l'outil, vérifie les données, et renvoie le résultat au LLM ;
4. le LLM formule la réponse finale.

Le LLM ne fait que **demander**. On le simule ici avec des règles simples, pour voir chaque étape.

In [ ]:
# --- L'agenda simulé de Baobab Software
AGENDA = {"lundi": ["09:00"], "mardi": ["10:00", "15:00"], "mercredi": [], "jeudi": ["11:00", "16:30"]}
JOURS = list(AGENDA)

# --- Un outil : une simple fonction Python avec une docstring claire
def check_availability(day: str) -> list:
    """Renvoie les créneaux libres d'un commercial pour un jour donné."""
    if day not in AGENDA:                       # validation côté serveur
        raise ValueError(f"Jour inconnu : {day}")
    return AGENDA[day]

# --- Génération automatique du schéma JSON, comme le fait @function_tool
def schema_outil(fn):
    types = {str: "string", int: "integer", float: "number", bool: "boolean"}
    sig = inspect.signature(fn)
    return {
        "type": "function",
        "name": fn.__name__,
        "description": inspect.getdoc(fn),
        "parameters": {
            "type": "object",
            "properties": {p: {"type": types.get(v.annotation, "string")} for p, v in sig.parameters.items()},
            "required": list(sig.parameters),
        },
    }

OUTILS = {"check_availability": check_availability}
print(json.dumps(schema_outil(check_availability), indent=2, ensure_ascii=False))

In [ ]:
def oral(heure):
    """'10:00' -> '10 h', '16:30' -> '16 h 30' : on écrit pour l'oreille, pas pour l'écran."""
    h, m = heure.split(":")
    return f"{int(h)} h" + ("" if m == "00" else f" {m}")

def faux_llm(messages, outils):
    """Imite les décisions d'un LLM avec des règles simples (pour la démonstration uniquement)."""
    dernier = messages[-1]
    if dernier["role"] == "tool":
        res = json.loads(dernier["content"])
        if dernier["name"] == "check_availability":
            if not res:
                return {"content": "Je n'ai plus de créneau ce jour-là. Voulez-vous un autre jour ?"}
            return {"content": f"J'ai {' ou '.join(oral(h) for h in res)}. Lequel vous convient ?"}
        if dernier["name"] == "book_meeting":
            return {"content": f"C'est réservé, référence {res['reference']}. À bientôt !"}
    texte = dernier["content"].lower()
    heure = re.search(r"(\d{1,2})\s*h", texte)
    if heure and "book_meeting" in outils:
        jour = next((m["args"]["day"] for m in reversed(messages) if m.get("args")), None)
        return {"tool_call": {"name": "book_meeting", "arguments": {"day": jour, "time": f"{int(heure.group(1)):02d}:00"}}}
    jour = next((j for j in JOURS if j in texte), None)
    if jour and "check_availability" in outils:
        return {"tool_call": {"name": "check_availability", "arguments": {"day": jour}}}
    return {"content": "Avec plaisir. Quel jour vous conviendrait pour un rendez-vous ?"}

def tour_de_parole(messages, texte_prospect, verbeux=True):
    """Un tour complet : prospect -> LLM -> (outil -> LLM) -> réponse d'Awa."""
    messages.append({"role": "user", "content": texte_prospect})
    if verbeux: print(f"Prospect : {texte_prospect}")
    while True:
        decision = faux_llm(messages, OUTILS)
        if "tool_call" in decision:
            nom, args = decision["tool_call"]["name"], decision["tool_call"]["arguments"]
            if verbeux: print(f"   LLM demande : {nom}({args})")
            resultat = OUTILS[nom](**args)                      # c'est NOTRE code qui exécute
            if verbeux: print(f"   Notre code exécute et renvoie : {resultat}")
            messages.append({"role": "tool", "name": nom, "args": args, "content": json.dumps(resultat)})
        else:
            messages.append({"role": "assistant", "content": decision["content"]})
            if verbeux: print(f"Awa : {decision['content']}\n")
            return decision["content"]

conversation = []
tour_de_parole(conversation, "Bonjour, je voudrais un rendez-vous mardi.")
tour_de_parole(conversation, "Et mercredi ?")

**[Guidé] Exercice 1.4** : donnez à Awa la capacité de **réserver**. Écrivez l'outil `book_meeting(day, time)` qui :
- vérifie que le créneau est libre (sinon `ValueError`) ;
- retire le créneau de l'agenda ;
- renvoie `{"reference": ...}` avec une référence du type `"BAO-mardi-10:00"`.

Puis enregistrez-le dans `OUTILS` et relancez la conversation.

In [ ]:
def book_meeting(day: str, time: str) -> dict:
    """Réserve un rendez-vous avec un commercial et renvoie sa référence."""
    # TODO 1 : si time n'est pas dans AGENDA.get(day, []), lever ValueError
    # TODO 2 : retirer le créneau de AGENDA[day]
    # TODO 3 : renvoyer {"reference": f"BAO-{day}-{time}"}
    raise NotImplementedError("Complétez book_meeting")

# TODO 4 : enregistrer l'outil dans OUTILS

try:
    conversation = []
    tour_de_parole(conversation, "Bonjour, je voudrais un rendez-vous mardi.")
    tour_de_parole(conversation, "Je prends 10 h.")
    print("Agenda de mardi après réservation :", AGENDA["mardi"])
except NotImplementedError as e:
    print(e)
if "book_meeting" not in OUTILS:
    print("Astuce : Awa ne sait pas encore réserver. Enregistrez book_meeting dans OUTILS (TODO 4).")

**[Avancé] Exercice 1.4 bis : l'idempotence.** Au téléphone, une coupure réseau peut faire **rejouer** un appel d'outil. Si `book_meeting` est appelé deux fois avec les mêmes arguments, il ne doit pas échouer ni réserver deux fois : il doit renvoyer la **même** référence.

Écrivez `book_meeting_idempotent` à l'aide d'un dictionnaire `RESERVATIONS` qui mémorise les réservations déjà faites.

In [ ]:
RESERVATIONS = {}
AGENDA["jeudi"] = ["11:00", "16:30"]

def book_meeting_idempotent(day: str, time: str) -> dict:
    """Réserve un rendez-vous ; un second appel identique renvoie la même référence."""
    # TODO : si (day, time) est déjà dans RESERVATIONS, renvoyer la réservation existante
    # sinon, réserver comme book_meeting et mémoriser le résultat
    raise NotImplementedError("Exercice avancé à compléter")

try:
    r1 = book_meeting_idempotent("jeudi", "11:00")
    r2 = book_meeting_idempotent("jeudi", "11:00")   # appel rejoué
    print(r1, r2, "->", "idempotent" if r1 == r2 else "problème")
except NotImplementedError as e:
    print(e)

### 1.5 [Avancé, optionnel] Mesurer un vrai LLM

Si une clé `OPENAI_API_KEY` est définie, la cellule mesure le **temps jusqu'au premier token** (TTFT) et le temps total d'un vrai modèle, en streaming. Sinon, elle ne fait rien.

*Question* : quelle part du temps total représente le TTFT ? Pourquoi est-ce le chiffre qui compte pour un agent vocal ?

In [ ]:
if not CLE_OPENAI:
    print("Pas de clé OPENAI_API_KEY : section ignorée (aucun problème pour la suite).")
else:
    from openai import OpenAI
    client = OpenAI()
    for modele in ["gpt-4.1-mini"]:
        t0 = time.perf_counter(); ttft = None; texte = ""
        flux = client.chat.completions.create(
            model=modele, stream=True,
            messages=[{"role": "system", "content": "Tu es Awa, assistante commerciale. Réponds en une phrase courte."},
                      {"role": "user", "content": "Je voudrais un rendez-vous mardi."}])
        for morceau in flux:
            delta = morceau.choices[0].delta.content if morceau.choices else None
            if delta:
                ttft = ttft or time.perf_counter() - t0
                texte += delta
        total = time.perf_counter() - t0
        print(f"{modele} : premier token {ttft:.2f} s, total {total:.2f} s -> {texte!r}")

### À retenir

- Un agent vocal **écoute** (STT), **comprend et agit** (LLM + outils), **répond** (TTS), en boucle.
- Le **streaming** ne réduit pas le temps total, il avance le **premier son** : c'est lui que l'utilisateur ressent.
- Des **phrases courtes** dans les réponses aident le streaming.
- Un budget de latence se pilote poste par poste, et se juge sur le **p95**, pas seulement sur la moyenne.
- Dans un appel d'outil, le **LLM demande**, **notre code exécute**, valide et renvoie le résultat. Les outils doivent être **idempotents**.

---
### Quiz du Module 1

Pour chaque question, écrivez la lettre de votre réponse dans le dictionnaire `mes_reponses`, puis exécutez la cellule de correction.

**Q1.** Quelle brique décide quand Awa a le droit de parler ?
a) le STT · b) la détection de fin de tour · c) le TTS · d) le serveur média

**Q2.** Quel est le principal avantage d'une architecture en cascade par rapport au speech-to-speech ?
a) la latence minimale · b) le texte visible à chaque étape et des briques interchangeables · c) l'intonation préservée · d) aucun fournisseur nécessaire

**Q3.** Le délai que ressent l'utilisateur se mesure :
a) du début de sa phrase à la fin de la réponse · b) de la fin de sa phrase au premier son de l'agent · c) par la durée totale de génération du LLM · d) par la durée de la transcription

**Q4.** Dans un appel d'outil `check_availability`, qui exécute réellement la requête sur l'agenda ?
a) le LLM · b) le TTS · c) notre code, côté serveur · d) le navigateur du prospect

**Q5.** Awa met 2,5 s à répondre. Le chronogramme montre que l'essentiel vient d'une fin de tour réglée de façon très prudente. Que faites-vous en premier ?
a) changer de TTS · b) réduire l'attente de fin de tour et ajouter une détection sémantique · c) passer à un plus grand LLM · d) augmenter le volume

In [ ]:
mes_reponses = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses ("a", "b", "c" ou "d")

In [ ]:
def corriger_quiz_module1(reponses):
    attendu = {1: "b", 2: "b", 3: "b", 4: "c", 5: "b"}
    explications = {
        1: "La détection de fin de tour (VAD + modèle sémantique) décide quand l'utilisateur a fini.",
        2: "En cascade, chaque brique est remplaçable et on voit le texte à chaque étape : idéal pour évaluer.",
        3: "Le délai perçu va de la dernière syllabe de l'utilisateur au premier son entendu.",
        4: "Le LLM ne fait que demander l'appel ; notre code l'exécute, le valide et renvoie le résultat.",
        5: "Le plus gros poste d'abord : une fin de tour trop prudente rend l'agent lent.",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module1(mes_reponses)

---
## 2. Le son numérique

**Objectifs**
1. Comprendre l'**échantillonnage**, la limite de **Nyquist** et le **repliement**.
2. Entendre et voir ce que le **téléphone (8 kHz)** fait à une voix.
3. Calculer un **débit** et comparer quantification linéaire et **µ-law** (G.711).
4. Découper l'audio en **trames** et repérer la parole par son énergie.
5. Mesurer l'effet du **bruit** et de l'**écho**.

**Durée** : 25 minutes, puis le quiz (5 minutes). Mettez vos écouteurs.

In [ ]:
# Cellule de reprise du Module 2
import numpy as np, matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")

def ecouter(x, fs, titre=""):
    """Affiche un lecteur audio (normalisé)."""
    if titre:
        print(titre)
    display(Audio(x, rate=fs, normalize=True))

print("Module 2 prêt.")

### 2.1 Échantillonner et replier

Échantillonner, c'est mesurer l'onde `fs` fois par seconde. Le **théorème de Nyquist-Shannon** dit qu'on ne peut capturer correctement que les fréquences inférieures à **fs / 2**. Au-delà, la fréquence se **replie** : on « voit » une fausse fréquence plus basse.

In [ ]:
def montrer_echantillonnage(f, fs, duree=0.004):
    tc = np.linspace(0, duree, 4000)
    ts = np.arange(0, duree, 1 / fs)
    plt.figure(figsize=(9, 2.6))
    plt.plot(tc * 1000, np.sin(2 * np.pi * f * tc), color=GRAY, lw=2, label=f"onde réelle {f} Hz")
    plt.stem(ts * 1000, np.sin(2 * np.pi * f * ts), linefmt=TEAL, markerfmt="o", basefmt=" ")
    if f >= fs / 2:
        fa = f - fs * round(f / fs)
        plt.plot(tc * 1000, np.sin(2 * np.pi * fa * tc), color="#C0392B", ls="--", lw=1.5,
                 label=f"fausse onde perçue {abs(fa):.0f} Hz")
    statut = "correct" if f < fs / 2 else "REPLIEMENT"
    plt.title(f"f = {f} Hz échantillonnée à {fs} Hz : {statut}", loc="left", fontweight="bold")
    plt.xlabel("temps (ms)"); plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
    plt.tight_layout(); plt.show()

montrer_echantillonnage(1000, 16000)
montrer_echantillonnage(1000, 1250)

**[Guidé] Exercice 2.1**
1. Complétez `nyquist` : la fréquence maximale capturable pour chaque fréquence d'échantillonnage.
2. Complétez `frequence_apparente(f, fs)` : pour une fréquence `f` échantillonnée **sans filtre** à `fs`, la fréquence perçue est `|f - fs * round(f / fs)|`.

*Question* : un « s » à 5 000 Hz passe dans un système à 8 kHz **sans filtre anti-repliement**. Quelle fréquence entend-on ?

In [ ]:
frequences = [8000, 16000, 24000, 48000]
nyquist = None   # TODO : dictionnaire {fs: fs / 2} pour chaque fs de la liste

def frequence_apparente(f, fs):
    return None  # TODO : appliquer la formule de l'énoncé

if nyquist is None or frequence_apparente(5000, 8000) is None:
    print("Complétez la cellule ci-dessus.")
else:
    for fs_, fmax in nyquist.items():
        print(f"{fs_:>6} Hz -> capture jusqu'à {fmax:.0f} Hz")
    print(f"Un « s » à 5000 Hz échantillonné à 8000 Hz sans filtre s'entend à {frequence_apparente(5000, 8000):.0f} Hz.")

### 2.2 La voix au téléphone

On fabrique une **voix de synthèse** : des voyelles (harmoniques sous 3 kHz) et des consonnes « s » (bruit entre 4,5 et 7,5 kHz). Puis on la passe en qualité web (16 kHz) et téléphone (8 kHz).

`signal.resample_poly` applique automatiquement un **filtre anti-repliement** avant de réduire la fréquence.

In [ ]:
def voyelle(duree, fs, f0=140, formants=((700, 1), (1200, 0.6), (2600, 0.25))):
    t = np.arange(int(duree * fs)) / fs
    phase = 2 * np.pi * np.cumsum(f0 * (1 + 0.04 * np.sin(2 * np.pi * 3 * t))) / fs
    x = np.zeros_like(t)
    for k in range(1, int(5000 / f0)):
        gain = sum(a * np.exp(-((k * f0 - F) / 180) ** 2) for F, a in formants) + 0.02
        x += gain * np.sin(k * phase)
    return x * np.minimum(1, np.minimum(t / 0.03, (duree - t) / 0.04).clip(0))

def fricative(duree, fs, rng, bande=(4500, 7500)):
    n = int(duree * fs)
    b, a = signal.butter(4, [bande[0] / (fs / 2), bande[1] / (fs / 2)], btype="band")
    x = signal.lfilter(b, a, rng.normal(size=n))
    env = np.minimum(1, np.minimum(np.arange(n), n - np.arange(n)) / (0.02 * fs))
    return 0.9 * x / np.max(np.abs(x)) * env

def voix_synthetique(fs=48000, seed=0):
    rng = np.random.default_rng(seed)
    z = lambda d: np.zeros(int(d * fs))
    parts = [z(0.15), fricative(0.12, fs, rng), voyelle(0.22, fs, 140),
             fricative(0.10, fs, rng), voyelle(0.20, fs, 155, ((300, 1), (2300, 0.5), (3000, 0.3))), z(0.18),
             fricative(0.12, fs, rng), voyelle(0.25, fs, 130, ((500, 1), (900, 0.7), (2500, 0.2))),
             fricative(0.15, fs, rng), z(0.2)]
    x = np.concatenate(parts)
    return 0.8 * x / np.max(np.abs(x))

x48 = voix_synthetique(48000)
x16 = signal.resample_poly(x48, 1, 3)   # 48 kHz -> 16 kHz
x8 = signal.resample_poly(x48, 1, 6)    # 48 kHz -> 8 kHz
ecouter(x16, 16000, "Qualité web (16 kHz)")
ecouter(x8, 8000, "Qualité téléphone (8 kHz)")

In [ ]:
def comparer_spectrogrammes(signaux, titre_general=""):
    """signaux = liste de (x, fs, titre)."""
    fig, axs = plt.subplots(1, len(signaux), figsize=(5.2 * len(signaux), 3.2), sharey=True)
    for ax, (x, fs, titre) in zip(np.atleast_1d(axs), signaux):
        n = 512 if fs >= 16000 else 256
        with np.errstate(divide="ignore"):
            ax.specgram(x + 1e-9, NFFT=n, Fs=fs, noverlap=int(n * 0.75), cmap="GnBu", vmin=-110)
        ax.set_ylim(0, 8000); ax.set_title(titre, loc="left", fontweight="bold"); ax.set_xlabel("temps (s)")
    np.atleast_1d(axs)[0].set_ylabel("fréquence (Hz)")
    plt.suptitle(titre_general, x=0.01, ha="left", fontweight="bold"); plt.tight_layout(); plt.show()

comparer_spectrogrammes([(x16, 16000, "Web, 16 kHz"), (x8, 8000, "Téléphone, 8 kHz")])

**[Guidé] Exercice 2.2** : complétez `part_energie_au_dessus` qui calcule la **part de l'énergie** d'un signal située au-dessus d'une fréquence donnée.

*Indice* : `spectre = np.abs(np.fft.rfft(x)) ** 2` et `freqs = np.fft.rfftfreq(len(x), 1 / fs)`. La part vaut la somme de `spectre` là où `freqs > fmin`, divisée par la somme totale.

In [ ]:
def part_energie_au_dessus(x, fs, fmin=4000):
    return None   # TODO

r16 = part_energie_au_dessus(x16, 16000)
if r16 is None:
    print("Complétez la fonction ci-dessus.")
else:
    r8 = part_energie_au_dessus(x8, 8000)
    print(f"Énergie au-dessus de 4 kHz : {r16:.1%} en 16 kHz, {r8:.1%} en 8 kHz.")

### 2.3 PCM, débit et µ-law

Le **PCM** est l'audio brut : un entier par échantillon. Son débit vaut `fréquence × bits × canaux`.

**[Guidé] Exercice 2.3** : complétez `debit_kbps` puis lisez le tableau.

In [ ]:
def debit_kbps(fs, bits, canaux=1):
    return None   # TODO : fs * bits * canaux, converti en kbit/s

formats = [("PCM 16 kHz, 16 bits", 16000, 16), ("PCM 8 kHz, 16 bits", 8000, 16),
           ("G.711 (8 kHz, 8 bits)", 8000, 8), ("PCM 48 kHz, 16 bits", 48000, 16)]
if debit_kbps(16000, 16) is None:
    print("Complétez la fonction ci-dessus.")
else:
    for nom, fs_, b in formats:
        d = debit_kbps(fs_, b)
        print(f"{nom:<24} {d:>6.0f} kbit/s   {d * 60 / 8 / 1000:>5.2f} Mo par minute")
    print("Opus (WebRTC) transporte souvent la voix entre 16 et 32 kbit/s.")

**Quantification linéaire contre µ-law.** G.711 code chaque échantillon sur **8 bits**, mais avec une échelle **logarithmique** (loi µ) qui donne plus de niveaux aux sons faibles. On mesure la qualité par le **rapport signal sur bruit** (SNR, en dB) : plus il est élevé, mieux c'est.

In [ ]:
MU = 255

def quantifier_lineaire(x, bits=8):
    L = 2 ** bits
    return np.round((np.clip(x, -1, 1) + 1) / 2 * (L - 1)) / (L - 1) * 2 - 1

def mulaw(x, bits=8):
    y = np.sign(x) * np.log1p(MU * np.abs(np.clip(x, -1, 1))) / np.log1p(MU)   # compression
    yq = quantifier_lineaire(y, bits)                                            # quantification 8 bits
    return np.sign(yq) * np.expm1(np.abs(yq) * np.log1p(MU)) / MU                # expansion

def snr_db(propre, degrade):
    return 10 * np.log10(np.sum(propre ** 2) / np.sum((propre - degrade) ** 2))

voix8 = x8 * 0.3   # une voix d'intensité moyenne
print(f"8 bits linéaire : SNR {snr_db(voix8, quantifier_lineaire(voix8)):.1f} dB")
print(f"8 bits µ-law    : SNR {snr_db(voix8, mulaw(voix8)):.1f} dB")
ecouter(quantifier_lineaire(voix8), 8000, "8 bits linéaire")
ecouter(mulaw(voix8), 8000, "8 bits µ-law (G.711)")

**[Avancé] Exercice 2.3 bis** : refaites la comparaison pour une voix **très faible** (`x8 * 0.02`, un prospect loin du micro), puis tracez le SNR des deux méthodes pour des gains de 0,01 à 1. Que montre la courbe sur l'intérêt de la loi µ ?

In [ ]:
# [Avancé] Exercice 2.3 bis : à vous

### 2.4 Trames et énergie

Toute la chaîne traite l'audio par **trames** de 10 à 30 ms. On découpe la voix à 16 kHz en trames de **20 ms** et on calcule l'**énergie** de chacune, en décibels. Les trames de parole sont nettement plus énergétiques que les silences.

In [ ]:
def trames(x, fs, duree_ms=20):
    n = int(fs * duree_ms / 1000)
    return x[: len(x) // n * n].reshape(-1, n)

def energie_db(tr):
    return 10 * np.log10(np.mean(tr ** 2, axis=1) + 1e-10)

T16 = trames(x16, 16000)
E16 = energie_db(T16)
print(f"{T16.shape[0]} trames de {T16.shape[1]} échantillons ({T16.shape[1] * 2} octets en PCM 16 bits).")

def tracer_energie(E, seuil=None, parole=None, titre="Énergie par trame de 20 ms"):
    plt.figure(figsize=(10, 2.8))
    plt.bar(np.arange(len(E)) * 20, E - E.min(), width=18, bottom=E.min(),
            color=[TEAL if (parole is not None and p) else GRAY for p in (parole if parole is not None else [0] * len(E))])
    if seuil is not None:
        plt.axhline(seuil, color="#C0392B", ls="--", label=f"seuil {seuil} dB"); plt.legend()
    plt.xlabel("temps (ms)"); plt.ylabel("dB"); plt.title(titre, loc="left", fontweight="bold"); plt.tight_layout(); plt.show()

tracer_energie(E16)

**[Guidé] Exercice 2.4** : choisissez un `SEUIL` (en dB) qui sépare la parole du silence, puis calculez `parole`, un tableau de booléens vrai pour les trames au-dessus du seuil. C'est un **détecteur d'activité vocale** (VAD) très simple.

**[Avancé] Exercice 2.4 bis** : un VAD réel ajoute une **traîne** (*hangover*) : il garde l'état « parole » quelques trames après la dernière trame forte, pour ne pas couper les fins de mots. Implémentez une traîne de 3 trames.

In [ ]:
SEUIL = None   # TODO : par exemple une valeur entre -60 et -20
parole = None  # TODO : E16 > SEUIL

if parole is None:
    print("Complétez la cellule ci-dessus.")
else:
    print(f"Trames de parole : {parole.sum()} sur {len(parole)} ({parole.mean():.0%}).")
    tracer_energie(E16, SEUIL, parole, "VAD par énergie")

### 2.5 Bruit et écho

Le **rapport signal sur bruit** (SNR) mesure en dB l'écart entre la puissance de la voix et celle du bruit : 20 dB, c'est un bureau calme ; 5 dB, c'est un marché animé.

**[Guidé] Exercice 2.5** : complétez `ajouter_bruit(x, snr_db)`. *Indice* : la puissance du bruit vaut `puissance_voix / 10 ** (snr_db / 10)`, et son écart-type est la racine carrée de cette puissance.

In [ ]:
rng = np.random.default_rng(1)

def ajouter_bruit(x, snr_db):
    p_voix = np.mean(x ** 2)
    sigma = None   # TODO : écart-type du bruit
    if sigma is None:
        return None
    return x + rng.normal(0, sigma, size=len(x))

def ajouter_echo(x, fs, retard_ms=120, gain=0.5):
    d = int(fs * retard_ms / 1000)
    y = x.copy(); y[d:] += gain * x[:-d]
    return y

bruite = ajouter_bruit(x16, 5)
if bruite is None:
    print("Complétez ajouter_bruit.")
else:
    ecouter(bruite, 16000, "Avec bruit de fond (SNR 5 dB)")
    ecouter(ajouter_echo(x16, 16000), 16000, "Avec écho (120 ms)")
    Eb = energie_db(trames(bruite, 16000))
    seuil = SEUIL if SEUIL is not None else -40
    print(f"Avec le bruit, le VAD par énergie détecte de la parole sur {(Eb > seuil).mean():.0%} des trames "
          f"(contre {(E16 > seuil).mean():.0%} sans bruit).")
    tracer_energie(Eb, seuil, Eb > seuil, "Le VAD par énergie face au bruit")

### 2.6 Atelier : votre propre voix

Enregistrez la phrase de test en **WAV** :
> « Je m'appelle Fatou Sow, mon numéro est le 77 616 06 16, et mon e-mail fatou.sow@baobab.sn. »

Indiquez le chemin du fichier dans `FICHIER`, puis exécutez la cellule. Sans fichier, la voix de synthèse est utilisée.

In [ ]:
from scipy.io import wavfile

FICHIER = None   # par exemple "ma_voix.wav"

if FICHIER:
    fs_orig, brut = wavfile.read(FICHIER)
    brut = brut.astype(np.float32)
    if brut.ndim > 1:
        brut = brut.mean(axis=1)                       # stéréo -> mono
    brut /= np.max(np.abs(brut)) + 1e-9
    v16 = signal.resample_poly(brut, 16000, fs_orig)
    v8 = signal.resample_poly(brut, 8000, fs_orig)
    print(f"Fichier chargé : {fs_orig} Hz, {len(brut) / fs_orig:.1f} s")
else:
    v16, v8 = x16, x8
    print("Aucun fichier : voix de synthèse utilisée.")

ecouter(v16, 16000, "Votre voix, 16 kHz"); ecouter(v8, 8000, "Votre voix, 8 kHz (téléphone)")
comparer_spectrogrammes([(v16, 16000, "16 kHz"), (v8, 8000, "8 kHz")])
r = part_energie_au_dessus(v16, 16000)
if r is not None:
    print(f"Part de votre énergie vocale au-dessus de 4 kHz : {r:.1%} (perdue au téléphone).")

### À retenir

- Numériser = **échantillonner** (fréquence) puis **quantifier** (bits) : c'est le **PCM**.
- On capture au mieux **fs / 2** (Nyquist). Au-delà, sans filtre, les fréquences se **replient**.
- Le téléphone (8 kHz) coupe tout au-delà de **4 kHz** : les sifflantes, donc les noms et les chiffres, souffrent. Rééchantillonner ensuite ne les fait pas revenir.
- Débit PCM = **fréquence × bits × canaux** ; les codecs (G.711, Opus) compressent pour le réseau.
- L'audio circule en **trames** de 10 à 30 ms ; un VAD par énergie est simple mais fragile face au **bruit**.

---
### Quiz du Module 2

**Q1.** Un système échantillonné à 8 kHz capture au mieux les fréquences jusqu'à :
a) 8 kHz · b) 4 kHz · c) 16 kHz · d) 2 kHz

**Q2.** On rééchantillonne un appel téléphonique de 8 kHz à 16 kHz avant de l'envoyer au STT. Les aigus perdus :
a) sont restaurés · b) ne sont pas restaurés · c) sont doublés · d) deviennent du bruit

**Q3.** Le débit d'un flux PCM 16 kHz, 16 bits, mono est de :
a) 16 kbit/s · b) 64 kbit/s · c) 256 kbit/s · d) 512 kbit/s

**Q4.** Quel codec est le standard de WebRTC pour la voix dans le navigateur ?
a) G.711 · b) Opus · c) MP3 · d) PCM 48 kHz

**Q5.** Quand le prospect utilise le haut-parleur de son téléphone, Awa s'interrompt elle-même au milieu de ses phrases. Que vérifiez-vous en premier ?
a) augmenter le volume · b) changer de STT · c) activer l'annulation d'écho · d) passer en 8 kHz

In [ ]:
mes_reponses_m2 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module2(reponses):
    attendu = {1: "b", 2: "b", 3: "c", 4: "b", 5: "c"}
    explications = {
        1: "Nyquist : on capture jusqu'à la moitié de la fréquence d'échantillonnage, soit 4 kHz.",
        2: "Ce qui n'a pas été capturé ne peut pas être reconstruit : rééchantillonner n'ajoute pas d'information.",
        3: "16 000 × 16 × 1 = 256 000 bit/s.",
        4: "Opus est le codec audio de référence de WebRTC ; G.711 est celui de la téléphonie classique.",
        5: "La voix d'Awa revient dans le micro du prospect : c'est de l'écho, d'où l'annulation d'écho (AEC).",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module2(mes_reponses_m2)

---
## 3. Le transport temps réel

**Objectifs**
1. Mesurer, avec un **vrai serveur WebSocket local**, ce que le streaming change au délai.
2. Comprendre pourquoi la voix préfère **UDP** à **TCP** (blocage en tête de file).
3. Régler un **tampon de gigue** et lire le compromis latence contre pertes.
4. **Entendre** l'effet des pertes de paquets et les masquer.
5. Calculer un **budget réseau** depuis Dakar.

**Durée** : 30 minutes, puis le quiz (8 minutes).

In [ ]:
# Cellule de reprise du Module 3
# !pip install -q websockets
import asyncio, json, time
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Audio, display
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
try:
    from websockets.asyncio.server import serve
    from websockets.asyncio.client import connect
except ImportError:
    raise RuntimeError("Installez la bibliothèque : pip install websockets")
RED = "#C0392B"
print("Module 3 prêt.")

### 3.1 Requête contre streaming, sur un vrai réseau local

On lance un **serveur WebSocket** sur votre machine. Il imite un service STT :
- en mode **lot** (*batch*), il reçoit tout l'audio d'un coup, puis le transcrit en 300 ms ;
- en mode **streaming**, il transcrit au fil de l'eau et renvoie des résultats **partiels**, puis un résultat final 50 ms après la dernière trame.

Le prospect parle pendant 1,8 s, soit 90 trames de 20 ms. On mesure le délai entre la **fin de sa phrase** et la **transcription finale**.

In [ ]:
TRAITEMENT_LOT = 0.30       # s : transcrire toute la phrase d'un coup
TRAITEMENT_FINAL = 0.05     # s : finaliser en streaming (le reste est déjà transcrit)
TRAME_S, N_TRAMES = 0.020, 90
TRAME = bytes(640)          # 20 ms de PCM 16 bits à 16 kHz (contenu sans importance ici)

async def stt_factice(ws):
    """Serveur : imite un service STT en lot ou en streaming."""
    config = json.loads(await ws.recv())
    recues = 0
    async for message in ws:
        if isinstance(message, bytes):
            recues += len(message) // len(TRAME)
            if config["mode"] == "stream" and recues % 15 == 0:
                await ws.send(json.dumps({"type": "partiel", "trames": recues}))
        elif json.loads(message)["type"] == "fin":
            await asyncio.sleep(TRAITEMENT_LOT if config["mode"] == "batch" else TRAITEMENT_FINAL)
            await ws.send(json.dumps({"type": "final", "trames": recues}))
            return

serveur = await serve(stt_factice, "127.0.0.1", 0)       # port libre choisi automatiquement
PORT = serveur.sockets[0].getsockname()[1]
URL = f"ws://127.0.0.1:{PORT}"
print("Serveur STT factice à l'écoute sur", URL)

In [ ]:
async def client(mode):
    """Renvoie (délai après la fin de la phrase, nombre de partiels reçus)."""
    async with connect(URL) as ws:
        await ws.send(json.dumps({"mode": mode}))
        partiels = []

        async def recevoir():
            async for m in ws:
                evt = json.loads(m)
                if evt["type"] == "partiel":
                    partiels.append(time.perf_counter())
                else:
                    return time.perf_counter()

        tache = asyncio.create_task(recevoir())
        if mode == "batch":
            await asyncio.sleep(TRAME_S * N_TRAMES)          # on enregistre toute la phrase…
            fin_parole = time.perf_counter()
            await ws.send(TRAME * N_TRAMES)                   # …puis on l'envoie d'un bloc
        else:
            for _ in range(N_TRAMES):                         # on envoie chaque trame dès qu'elle existe
                await ws.send(TRAME)
                await asyncio.sleep(TRAME_S)
            fin_parole = time.perf_counter()
        await ws.send(json.dumps({"type": "fin"}))
        t_final = await tache
        return t_final - fin_parole, len(partiels)

d_lot, _ = await client("batch")
d_flux, n_part = await client("stream")
print(f"Lot       : transcription finale {d_lot * 1000:.0f} ms après la fin de la phrase")
print(f"Streaming : transcription finale {d_flux * 1000:.0f} ms après la fin de la phrase ({n_part} résultats partiels reçus pendant la parole)")

**[Guidé] Exercice 3.1** : les vrais services STT en lot sont souvent plus lents sur une phrase longue. Passez `TRAITEMENT_LOT` à 0,8 s, relancez les deux cellules du client et calculez le **gain** du streaming en millisecondes.

*Question* : pourquoi le délai du streaming ne dépend-il presque pas de la longueur de la phrase ?

In [ ]:
TRAITEMENT_LOT = 0.30   # TODO : passez à 0.8
d_lot, _ = await client("batch")
d_flux, _ = await client("stream")
gain_ms = None          # TODO : différence en millisecondes

print("Complétez la cellule." if gain_ms is None else f"Gain du streaming : {gain_ms:.0f} ms")

In [ ]:
serveur.close()
await serveur.wait_closed()
TRAITEMENT_LOT = 0.30
print("Serveur arrêté.")

### 3.2 TCP contre UDP

On envoie un paquet toutes les 20 ms. Le réseau ajoute un délai de base, une **gigue** aléatoire, et perd quelques paquets.
- **UDP** : un paquet perdu est simplement absent.
- **TCP** : un paquet perdu est **renvoyé** après un délai (RTO), et comme TCP livre **dans l'ordre**, tous les paquets suivants attendent derrière lui : c'est le **blocage en tête de file**.

In [ ]:
def simuler_reseau(n=500, base=40, gigue=12, perte=0.03, rto=200, seed=0):
    rng = np.random.default_rng(seed)
    envoi = np.arange(n) * 20.0
    arrivee = envoi + base + rng.gamma(2, gigue / 2, n)
    perdu = rng.random(n) < perte
    udp = np.where(perdu, np.nan, arrivee)
    tcp = arrivee.copy(); tcp[perdu] += rto
    tcp = np.maximum.accumulate(tcp)              # livraison dans l'ordre
    return envoi, udp - envoi, tcp - envoi, perdu

envoi, d_udp, d_tcp, perdu = simuler_reseau()
plt.figure(figsize=(10, 3.2))
plt.plot(envoi[:150], d_tcp[:150], color=AMBER, lw=2, label="TCP")
plt.scatter(envoi[:150], d_udp[:150], color=TEAL, s=14, zorder=3, label="UDP")
for i in np.where(perdu[:150])[0]:
    plt.axvline(envoi[i], color=RED, ls=":", lw=1)
plt.axhline(150, color=GRAY, ls="--"); plt.text(5, 155, "150 ms", color="gray")
plt.xlabel("instant d'envoi (ms)"); plt.ylabel("délai de livraison (ms)"); plt.legend()
plt.title("Les 3 premières secondes", loc="left", fontweight="bold"); plt.tight_layout(); plt.show()

**[Guidé] Exercice 3.2** : pour chaque mode, calculez le **délai moyen** des paquets livrés et la **part de paquets inutilisables**, c'est-à-dire perdus (UDP) ou livrés après 150 ms (les deux modes). Puis refaites le calcul avec 8 % de pertes, comme sur un mobile en zone saturée.

*Indice* : `np.nanmean`, `np.isnan`, et une comparaison `> 150`.

In [ ]:
def bilan(d_udp, d_tcp):
    moy_udp = None   # TODO : délai moyen UDP (en ignorant les NaN)
    moy_tcp = None   # TODO : délai moyen TCP
    inut_udp = None  # TODO : part de paquets perdus OU livrés après 150 ms
    inut_tcp = None  # TODO : part de paquets livrés après 150 ms
    return moy_udp, moy_tcp, inut_udp, inut_tcp

for p in (0.03, 0.08):
    r = bilan(*simuler_reseau(perte=p)[1:3])
    if None in r:
        print("Complétez la fonction bilan."); break
    print(f"pertes {p:.0%} | UDP : {r[0]:.0f} ms, inutilisables {r[2]:.1%} | TCP : {r[1]:.0f} ms, inutilisables {r[3]:.1%}")

### 3.3 Le tampon de gigue

En UDP, les paquets arrivent irrégulièrement. Le **tampon de gigue** attend un court instant avant de jouer chaque paquet. Un paquet qui arrive après `délai de base + tampon` est **jeté**, comme s'il était perdu.

**[Guidé] Exercice 3.3** : complétez `taux_retard`, tracez la courbe, puis trouvez le **plus petit tampon** qui garde les paquets jetés sous **2 %**.

In [ ]:
BASE = 40
_, d_udp, _, _ = simuler_reseau(n=5000, gigue=16, perte=0.0, seed=4)

def taux_retard(delais, tampon, base=BASE):
    return None   # TODO : part des délais supérieurs à base + tampon

tampons = np.arange(0, 121, 5)
if taux_retard(d_udp, 40) is None:
    print("Complétez taux_retard.")
else:
    taux = np.array([taux_retard(d_udp, t) for t in tampons])
    meilleur = tampons[np.argmax(taux < 0.02)]
    plt.figure(figsize=(8, 3))
    plt.plot(tampons, taux * 100, color=NAVY, marker="o", ms=3)
    plt.axhline(2, color=RED, ls="--"); plt.axvline(meilleur, color=TEAL, ls=":")
    plt.xlabel("tampon (ms)"); plt.ylabel("paquets jetés (%)"); plt.tight_layout(); plt.show()
    print(f"Plus petit tampon sous 2 % de paquets jetés : {meilleur} ms")

**[Avancé] Exercice 3.3 bis : un tampon adaptatif.** Le réseau change : calme, puis saturé, puis calme. Un tampon fixe est soit trop long au calme, soit trop court en pointe. Implémentez un tampon **adaptatif** : pour chaque paquet, la taille du tampon vaut le **99e percentile** des 50 derniers délais, moins la base. Comparez paquets jetés et tampon moyen avec un tampon fixe de 40 ms.

In [ ]:
rng = np.random.default_rng(9)
gigue_variable = np.r_[np.full(1500, 6), np.full(1000, 30), np.full(1500, 6)]
d_var = BASE + rng.gamma(2, gigue_variable / 2)

# TODO [Avancé] : calculer le tampon adaptatif paquet par paquet et comparer au tampon fixe

### 3.4 Les pertes à l'oreille

On supprime aléatoirement des **trames de 20 ms** d'une voix, comme le ferait le réseau, puis on écoute.

**[Guidé] Exercice 3.4** : complétez `perdre_trames` qui met à zéro une part `taux` des trames.

**[Avancé] Exercice 3.4 bis** : écrivez `masquer`, un **masquage de pertes** très simple : chaque trame perdue est remplacée par la trame précédente, atténuée de moitié. Écoutez la différence. Opus fait la même chose, en bien plus sophistiqué.

In [ ]:
FS = 16000
try:
    voix = x16                                   # voix du module 2 si disponible
except NameError:
    t = np.arange(int(1.6 * FS)) / FS
    voix = 0.5 * np.sin(2 * np.pi * 140 * t) * (np.sin(2 * np.pi * 2.5 * t) > 0)   # repli simple
N = int(0.020 * FS)
voix = voix[: len(voix) // N * N]

def perdre_trames(x, taux, seed=2):
    rng = np.random.default_rng(seed)
    y = x.copy().reshape(-1, N)
    perdues = rng.random(len(y)) < taux
    # TODO : mettre à zéro les trames perdues
    return y.reshape(-1), perdues

def masquer(x, perdues):
    return None   # TODO [Avancé]

for taux in (0.05, 0.15):
    y, perdues = perdre_trames(voix, taux)
    print(f"{taux:.0%} de trames perdues ({perdues.sum()} trames)")
    display(Audio(y, rate=FS, normalize=True))
    z = masquer(y, perdues)
    if z is not None:
        print("   avec masquage :"); display(Audio(z, rate=FS, normalize=True))

### 3.5 Le budget réseau depuis Dakar

La lumière parcourt environ **200 km par milliseconde** dans la fibre. C'est un plancher physique : les routes réelles sont plus longues que la ligne droite, et passent par des équipements. On estime ici le délai aller vers quelques régions où l'on peut héberger un agent.

*Distances à vol d'oiseau, calculées ; délais réels à mesurer (ping, statistiques WebRTC).*

In [ ]:
def distance_km(a, b):
    """Distance à vol d'oiseau (formule de haversine)."""
    (la1, lo1), (la2, lo2) = np.radians(a), np.radians(b)
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(h))

DAKAR = (14.69, -17.45)
REGIONS = {"Lagos": (6.52, 3.38), "Paris": (48.86, 2.35), "Francfort": (50.11, 8.68),
           "Johannesburg": (-26.20, 28.05), "Virginie (États-Unis)": (39.04, -77.49)}
FIBRE_KM_PAR_MS = 200
FACTEUR_ROUTE = 1.6        # hypothèse : la route réelle est environ 60 % plus longue

for nom, coord in REGIONS.items():
    d = distance_km(DAKAR, coord)
    print(f"{nom:<22} {d:>6.0f} km   propagation min {d / FIBRE_KM_PAR_MS:>4.0f} ms   estimation {d * FACTEUR_ROUTE / FIBRE_KM_PAR_MS:>4.0f} ms (aller)")

**[Guidé] Exercice 3.5** : le délai aller visé est de **150 ms** (recommandation G.114 de l'UIT). Complétez le budget d'un appel d'Awa hébergée à **Paris** et calculez la marge restante.

| Poste | Valeur |
|---|---|
| Réseau d'accès du prospect (4G) | 40 ms |
| Propagation estimée Dakar → Paris | voir 3.5 |
| Tampon de gigue | 40 ms |
| Encodage et décodage Opus (trame de 20 ms) | 25 ms |

In [ ]:
budget = {
    "acces_4g": 40,
    "propagation": None,   # TODO : estimation Dakar -> Paris calculée ci-dessus
    "tampon_gigue": 40,
    "codec": 25,
}
if budget["propagation"] is None:
    print("Complétez la propagation.")
else:
    total = sum(budget.values())
    print(f"Total aller : {total:.0f} ms   ->   marge sur 150 ms : {150 - total:.0f} ms")

### À retenir

- En temps réel, **mieux vaut perdre que attendre** : la voix voyage en **UDP** (WebRTC, RTP).
- **TCP** livre tout, dans l'ordre : une perte bloque tous les paquets suivants (blocage en tête de file).
- Le **streaming** WebSocket avance la transcription finale, car l'essentiel est fait pendant la parole.
- Le **tampon de gigue** échange de la latence contre moins de paquets jetés ; l'idéal est **adaptatif**.
- Le **masquage des pertes** rend quelques pourcents de pertes presque inaudibles.
- Le réseau consomme une bonne part des **150 ms** : héberger au plus près et **mesurer**.

---
### Quiz du Module 3

**Q1.** Pourquoi la voix en temps réel voyage-t-elle en UDP plutôt qu'en TCP ?
a) UDP est plus sûr · b) un paquet en retard est inutile : mieux vaut le perdre que l'attendre · c) UDP est plus simple à coder · d) UDP est chiffré par défaut

**Q2.** Entre le worker d'Awa et un service STT en streaming, quel protocole est le plus courant ?
a) SIP · b) SMTP · c) WebSocket · d) FTP

**Q3.** Dans un appel téléphonique, SIP sert à :
a) transporter la voix · b) établir, modifier et terminer l'appel · c) compresser l'audio · d) masquer les pertes

**Q4.** Que se passe-t-il quand on augmente la taille du tampon de gigue ?
a) la latence baisse · b) les pertes réseau disparaissent · c) le codec change · d) moins de paquets sont jetés, mais la latence augmente

**Q5.** Les prospects en 4G trouvent la voix d'Awa hachée, pas ceux en Wi-Fi. Que faites-vous en premier ?
a) relever pertes et gigue dans les statistiques WebRTC · b) changer de TTS · c) passer l'audio en 48 kHz · d) remplacer WebRTC par HTTP

In [ ]:
mes_reponses_m3 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module3(reponses):
    attendu = {1: "b", 2: "c", 3: "b", 4: "d", 5: "a"}
    explications = {
        1: "Une trame qui arrive 200 ms trop tard ne sert plus : UDP ne bloque pas le flux pour la renvoyer.",
        2: "WebSocket : une connexion ouverte dans les deux sens, idéale entre serveurs pour streamer l'audio.",
        3: "SIP est la signalisation (INVITE, 200 OK, BYE) ; la voix passe à côté, en RTP.",
        4: "C'est le compromis du tampon : moins de paquets jetés contre plus de latence.",
        5: "Le problème dépend du réseau : on mesure d'abord pertes et gigue avant de changer quoi que ce soit.",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module3(mes_reponses_m3)